# End-to-End Agricultural Data Mining Workflow

This notebook is the single reproducibility entry point for the project. It runs the complete workflow from the raw Stata modules through household-level preparation, leakage-safe classification, hyperparameter improvement, ANN comparison, and clustering.

The detailed notebooks remain available for explanation and inspection, while this runner prevents the final workflow from being scattered across disconnected manual steps.

## Workflow contract

- Input: raw files in `stata/`.
- Household key: `idquest`.
- Target: agricultural-input use from `s5q1_1`.
- Classification split: stratified 75/25 split with `random_state=42`.
- Predictors: household context, land/crops, agricultural practices, and aggregated livestock variables.
- Leakage control: direct `S5` response fields and `idquest` are excluded from predictors.
- Outputs: prepared data, audit files, model metrics, ANN results, and cluster artifacts.

In [ ]:
from pathlib import Path
import subprocess
import sys

root = Path.cwd()
if root.name != 'Mid-Sem Exam':
    root = next(path for path in [root, *root.parents] if (path / 'run_end_to_end.py').exists())

subprocess.run([sys.executable, 'run_end_to_end.py'], cwd=root, check=True)
print('End-to-end workflow completed.')

In [ ]:
import pandas as pd

prepared = pd.read_csv(root / 'prepared_household_data.csv')
baseline = pd.read_csv(root / 'phase_3_baseline_results.csv')
improvement = pd.read_csv(root / 'phase_4_improvement_results.csv')
final_models = pd.read_csv(root / 'final_model_comparison.csv')
clusters = pd.read_csv(root / 'phase_4_cluster_profiles.csv')

assert prepared['idquest'].is_unique
assert prepared['target'].isin([0, 1]).all()
assert len(baseline) == 3
assert len(final_models) == 5
assert improvement.loc[1, 'F1'] > improvement.loc[0, 'F1']
assert clusters['households'].sum() == len(prepared)

print('Prepared households:', len(prepared))
print('Baseline models:', len(baseline))
print('Final models:', len(final_models))
print('Best model:', final_models.iloc[0]['Model'])
print('Cluster profiles:')
display(clusters)

## Interpretation checkpoint

The final model table is the authoritative source for reported classification metrics. The cluster profile table must be interpreted with the documented imbalance limitation: a strong silhouette score does not by itself establish meaningful or balanced agricultural segments.